# Charging Infrastructure – Data Exploration

This notebook explores electric vehicle charging infrastructure across European countries.

The analysis focuses on the dataset structure, geographic and temporal coverage, data quality, and its suitability for investigating the relationship between charging infrastructure and EV adoption.

In [1]:
# Import libraries
import pandas as pd

# Load the dataset
df_charging = pd.read_csv("../data/raw/eurostat_charging_infrastructure.csv")

# Inspect dataset structure
print("Dataset shape:", df_charging.shape)

display(df_charging.head())

df_charging.info()

Dataset shape: (488, 10)


,DATAFLOW,LAST UPDATE,freq,tra_infr,geo,unit,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:ROAD_IF_RCHEV(1.0),28/10/26 23:00:00,Annual,Recharging point,Austria,Number,2021,9143.0,NaN,NaN
1,ESTAT:ROAD_IF_RCHEV(1.0),28/10/26 23:00:00,Annual,Recharging point,Austria,Number,2022,14526.0,NaN,NaN
2,ESTAT:ROAD_IF_RCHEV(1.0),28/10/26 23:00:00,Annual,Recharging point,Austria,Number,2023,19663.0,NaN,NaN
3,ESTAT:ROAD_IF_RCHEV(1.0),28/10/26 23:00:00,Annual,Recharging point,Austria,Number,2024,25632.0,NaN,NaN
4,ESTAT:ROAD_IF_RCHEV(1.0),28/10/26 23:00:00,Annual,Recharging point,Austria,Number,2025,34333.0,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 488 entries, 0 to 487
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATAFLOW     488 non-null    str    
 1   LAST UPDATE  488 non-null    str    
 2   freq         488 non-null    str    
 3   tra_infr     488 non-null    str    
 4   geo          488 non-null    str    
 5   unit         488 non-null    str    
 6   TIME_PERIOD  488 non-null    int64  
 7   OBS_VALUE    486 non-null    float64
 8   OBS_FLAG     47 non-null     str    
 9   CONF_STATUS  0 non-null      float64
dtypes: float64(2), int64(1), str(7)
memory usage: 77.2 KB


In [ ]:
# Check geographic and temporal coverage
print("Number of countries:", df_charging["geo"].nunique())
print("Countries:", sorted(df_charging["geo"].unique()))

print("\nYear range:")
print(df_charging["TIME_PERIOD"].min(), "-", df_charging["TIME_PERIOD"].max())

# Check categorical variables
for col in ["freq", "tra_infr", "unit"]:
    print(f"\n{col}:")
    print(df_charging[col].value_counts())

# Check duplicate country-year combinations
print(
    "\nDuplicate country-year combinations:",
    df_charging.duplicated(subset=["geo", "TIME_PERIOD"]).sum(),
)

Number of countries: 35
Countries: ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'Estonia', 'European Union - 27 countries (from 2020)', 'Finland', 'France', 'Georgia', 'Germany', 'Greece', 'Hungary', 'Iceland', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta', 'Netherlands', 'North Macedonia', 'Norway', 'Poland', 'Portugal', 'Romania', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Türkiye', 'United Kingdom']

Year range:
2019 - 2025

freq:
freq
Annual    488
Name: count, dtype: int64

tra_infr:
tra_infr
Recharging pool                                     172
Recharging point                                    158
Recharging station                                   98
Recharging pool dedicated to heavy-duty vehicles     60
Name: count, dtype: int64

unit:
unit
Number      367
Kilowatt    121
Name: count, dtype: int64

Duplicate country-year combinations: 312


In [ ]:
# Check duplicates using all dimensions that define an observation
duplicate_count = df_charging.duplicated(
    subset=["geo", "TIME_PERIOD", "tra_infr", "unit"]
).sum()

print("Duplicate country-year-infrastructure-unit combinations:", duplicate_count)

# Inspect the two missing observation values
display(
    df_charging.loc[
        df_charging["OBS_VALUE"].isna(),
        ["geo", "TIME_PERIOD", "tra_infr", "unit", "OBS_VALUE", "OBS_FLAG"],
    ]
)

Duplicate country-year-infrastructure-unit combinations: 0


,geo,TIME_PERIOD,tra_infr,unit,OBS_VALUE,OBS_FLAG
330,Switzerland,2025,Recharging pool dedicated to heavy-duty vehicles,Kilowatt,NaN,m
331,Switzerland,2025,Recharging pool dedicated to heavy-duty vehicles,Number,NaN,m


In [4]:
# Check missing values
print("Missing values:")
display(df_charging.isna().sum())

# Check observation flags
print("\nObservation flags:")
display(df_charging["OBS_FLAG"].value_counts(dropna=False))

# Check negative and zero values
print("\nNegative values:", (df_charging["OBS_VALUE"] < 0).sum())
print("Zero values:", (df_charging["OBS_VALUE"] == 0).sum())

# Summary of charging infrastructure values
display(df_charging["OBS_VALUE"].describe())

Missing values:


DATAFLOW         0
LAST UPDATE      0
freq             0
tra_infr         0
geo              0
unit             0
TIME_PERIOD      0
OBS_VALUE        2
OBS_FLAG       441
CONF_STATUS    488
dtype: int64


Observation flags:


OBS_FLAG
NaN    441
i       44
m        2
b        1
Name: count, dtype: int64


Negative values: 0
Zero values: 29


count    4.860000e+02
mean     3.091677e+05
std      2.204809e+06
min      0.000000e+00
25%      5.060000e+02
50%      3.188500e+03
75%      2.806050e+04
max      4.441730e+07
Name: OBS_VALUE, dtype: float64

### Data Quality – Charging Infrastructure

The dataset contains 488 observations covering annual charging infrastructure data from 2019 to 2025.

Two observations have missing values for Switzerland in 2025, both related to recharging pools dedicated to heavy-duty vehicles. These observations are retained as missing rather than replaced with zero.

No negative values or duplicate observations were found. The 29 zero values are retained as potentially valid observations.

The dataset includes different infrastructure categories and measurement units (Number and Kilowatt), which must be considered during analysis.

In [ ]:
# Inspect the largest charging infrastructure value
display(
    df_charging.loc[
        df_charging["OBS_VALUE"].idxmax(),
        ["geo", "TIME_PERIOD", "tra_infr", "unit", "OBS_VALUE"],
    ]
)

geo            European Union - 27 countries (from 2020)
TIME_PERIOD                                         2025
tra_infr                                 Recharging pool
unit                                            Kilowatt
OBS_VALUE                                   44417302.222
Name: 220, dtype: object

### Data Quality – Extreme Values

The maximum infrastructure value corresponds to the EU-27 aggregate in 2025, for recharging pools measured in kilowatts.

The large value reflects aggregated charging capacity rather than an individual country-level observation. Country-level analysis should exclude EU aggregates and distinguish between infrastructure counts and charging capacity.

In [6]:
# Inspect infrastructure types and measurement units
display(
    df_charging[["tra_infr", "unit"]]
    .drop_duplicates()
    .sort_values(["tra_infr", "unit"])
)

,tra_infr,unit
0,Recharging point,Number
158,Recharging pool,Kilowatt
164,Recharging pool,Number
330,Recharging pool dedicated to heavy-duty vehicles,Kilowatt
331,Recharging pool dedicated to heavy-duty vehicles,Number
390,Recharging station,Number


### Infrastructure Categories

The dataset contains several charging infrastructure indicators measured either as counts or as installed power in kilowatts.

These indicators represent different infrastructure concepts and cannot be directly combined. The main analysis will focus on passenger-car charging infrastructure, while heavy-duty infrastructure will be excluded.

The final indicator selection will be made during data cleaning after confirming the definitions of the infrastructure categories.